# Exploratory Data Analysis

Goal: understand the dataset before committing to a preprocessing strategy. Plots are saved to plots/ so they can be embedded in the README.

Sections:
1. Setup
2. Shape and target balance
3. Missing values
4. Transaction amount
5. Categorical features vs fraud rate
6. TransactionDT structure
7. V-block correlation
8. Decisions

## 1. Setup

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')
for p in ['.', '..', '/kaggle/working/ML_Asgn2']:
    if os.path.isdir(os.path.join(p, 'src')) and p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_utils import load_train, split_columns

sns.set_theme(style='whitegrid', context='talk')
PLOTS = 'plots'
os.makedirs(PLOTS, exist_ok=True)

## 2. Shape and target balance

In [ ]:
X, y = load_train(sample_frac=None)  # full train for EDA
print(f'rows: {len(X):,}  cols: {X.shape[1]}')
print(f'fraud rate: {y.mean():.4f}  ({y.sum():,} of {len(y):,} are fraud)')

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
vc = y.value_counts().sort_index()
ax.bar(['Legit (0)', 'Fraud (1)'], vc.values, color=['#4c72b0', '#c44e52'])
for i, v in enumerate(vc.values):
    ax.text(i, v, f'{v:,}\n({v / len(y):.2%})', ha='center', va='bottom')
ax.set_title('Class balance')
ax.set_ylabel('count')
fig.tight_layout()
fig.savefig(f'{PLOTS}/01_class_balance.png', dpi=120)
plt.show()

## 3. Missing values

In [ ]:
missing = X.isna().mean().sort_values(ascending=False)
print(f'columns with any NaN: {(missing > 0).sum()}')
print(f'columns >50% NaN:    {(missing > 0.5).sum()}')
print(f'columns >90% NaN:    {(missing > 0.9).sum()}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(missing.values, bins=40, color='#dd8452', edgecolor='white')
ax.set_xlabel('fraction of NaN per column')
ax.set_ylabel('number of columns')
ax.set_title('Missing-value distribution across columns')
fig.tight_layout()
fig.savefig(f'{PLOTS}/02_missing_distribution.png', dpi=120)
plt.show()

## 4. Transaction amount

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(X['TransactionAmt'].clip(upper=1000), bins=60, color='#4c72b0')
axes[0].set_title('TransactionAmt (clipped at 1000)')
axes[0].set_xlabel('amount')

axes[1].hist(np.log1p(X['TransactionAmt']), bins=60, color='#55a868')
axes[1].set_title('log1p(TransactionAmt)')
axes[1].set_xlabel('log amount')

fig.suptitle('Why we engineer TransactionAmt_log')
fig.tight_layout()
fig.savefig(f'{PLOTS}/03_transaction_amt.png', dpi=120)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
for cls, color, label in [(0, '#4c72b0', 'Legit'), (1, '#c44e52', 'Fraud')]:
    sns.kdeplot(np.log1p(X.loc[y == cls, 'TransactionAmt']), ax=ax,
                color=color, fill=True, alpha=0.4, label=label)
ax.set_xlabel('log1p(TransactionAmt)')
ax.set_title('log-amount distribution by class')
ax.legend()
fig.tight_layout()
fig.savefig(f'{PLOTS}/04_amt_by_class.png', dpi=120)
plt.show()

## 5. Categorical features vs fraud rate

In [ ]:
def fraud_rate_by(col, top_n=10):
    s = (pd.DataFrame({'cat': X[col].astype('object').fillna('missing'), 'y': y})
         .groupby('cat')['y'].agg(['mean', 'count'])
         .query('count >= 500')
         .sort_values('mean', ascending=False)
         .head(top_n))
    return s

for col in ['ProductCD', 'card4', 'card6', 'P_emaildomain']:
    if col not in X.columns:
        continue
    s = fraud_rate_by(col, top_n=10)
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.barh(s.index.astype(str), s['mean'], color='#c44e52')
    ax.axvline(y.mean(), color='black', ls='--', label=f'overall ({y.mean():.3f})')
    ax.set_xlabel('fraud rate')
    ax.set_title(f'Top categories of {col} by fraud rate (n>=500)')
    ax.legend()
    fig.tight_layout()
    fig.savefig(f'{PLOTS}/05_{col}_fraud_rate.png', dpi=120)
    plt.show()

## 6. TransactionDT structure

TransactionDT is a seconds-offset from a reference timestamp. Plotting by hour-of-day shows the daily cycle for fraud vs legit.

In [ ]:
hours = (X['TransactionDT'] / 3600) % 24
fig, ax = plt.subplots(figsize=(11, 4))
for cls, color, label in [(0, '#4c72b0', 'Legit'), (1, '#c44e52', 'Fraud')]:
    counts, _ = np.histogram(hours[y == cls], bins=24)
    ax.plot(range(24), counts / counts.sum(), color=color, label=label, lw=2)
ax.set_xlabel('hour-of-day (relative to reference)')
ax.set_ylabel('share of transactions')
ax.set_title('Diurnal pattern: legit vs fraud')
ax.legend()
fig.tight_layout()
fig.savefig(f'{PLOTS}/06_hour_of_day.png', dpi=120)
plt.show()

## 7. V-block correlation

The 339 anonymised V-features are heavily redundant. A heatmap of a small slice is enough to motivate correlation pruning during feature selection.

In [ ]:
v_sample = [c for c in X.columns if c.startswith('V')][:30]
corr = X[v_sample].corr().fillna(0)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1,
            cbar_kws={'shrink': 0.6}, ax=ax)
ax.set_title('Correlation among first 30 V-features')
fig.tight_layout()
fig.savefig(f'{PLOTS}/07_v_correlation.png', dpi=120)
plt.show()

## 8. Decisions

- Class imbalance (~3.5%) → optimise ROC-AUC, not accuracy.
- NaN structure carries signal → trees use sentinel fill (`-999`); linear models median-impute.
- TransactionAmt is heavy-tailed → engineer `TransactionAmt_log`.
- Email TLD has strong fraud signal → engineer `*_emaildomain_suffix`.
- V-features are redundant → correlation pruning is one of the FS candidates in tree notebooks.
- `P_emaildomain` / `R_emaildomain` have many rare values → linear notebooks cap OHE at 20 categories.